In [3]:
import sys
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_preprocessing import prepare_dataset

raw_df = pd.read_csv(ROOT / "data" / "raw" / "amazon_mobile_reviews.csv")
print("Raw dataset:", raw_df.shape)
print("Missing values before cleaning:")
display(raw_df.isna().sum())

# prepare_dataset normalizes text and labels, removes invalid rows and
# duplicate labeled reviews, and converts invalid ratings to missing values.
df = prepare_dataset(raw_df)

print("Cleaned dataset:", df.shape)
print("Rows removed:", len(raw_df) - len(df))
print("Sentiment counts after cleaning:")
display(df["sentiment"].value_counts())
if "rating" in df.columns:
    print("Missing or invalid ratings after cleaning:", df["rating"].isna().sum())
display(df.head())

Raw dataset: (20000, 5)
Missing values before cleaning:


review_id       0
product_name    0
review_text     0
rating          0
sentiment       0
dtype: int64

Cleaned dataset: (20000, 6)
Rows removed: 0
Sentiment counts after cleaning:


sentiment
Positive    14353
Negative     3358
Neutral      2289
Name: count, dtype: int64

Missing or invalid ratings after cleaning: 0


,review_id,product_name,review_text,rating,sentiment,clean_text
0,1,120401325X,They look good and stick good! I just don't li...,4.0,Positive,they look good and stick good! i just don't li...
1,2,120401325X,These stickers work like the review says they ...,5.0,Positive,these stickers work like the review says they ...
2,3,120401325X,These are awesome and make my phone look so st...,5.0,Positive,these are awesome and make my phone look so st...
3,4,120401325X,Item arrived in great time and was in perfect ...,4.0,Positive,item arrived in great time and was in perfect ...
4,5,120401325X,"awesome! stays on, and looks great. can be use...",5.0,Positive,"awesome! stays on, and looks great. can be use..."


In [4]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["sentiment"]
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["sentiment"]
)

output_dir = ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(output_dir / "train.csv", index=False)
validation_df.to_csv(output_dir / "validation.csv", index=False)
test_df.to_csv(output_dir / "test.csv", index=False)

print("Training records:", len(train_df))
print("Validation records:", len(validation_df))
print("Testing records:", len(test_df))

Training records: 16000
Validation records: 2000
Testing records: 2000
